# TasteTrend GenAI — End-to-End Demo Notebook

This notebook demonstrates the **full pipeline** from **ETL → OpenSearch → Bedrock Agent → Live Querying**.

### What you can do here:
1. Configure your environment (region, API URL, API key)
2. Trigger the ETL Lambda to process raw review data
3. Trigger the Embedding Lambda to index documents into OpenSearch
4. Query the deployed Bedrock Agent via API Gateway
5. Run evaluation metrics against the example business queries
6. Test live prompts interactively

### Architecture
```
S3 (raw) → ETL Lambda → S3 (processed) → Embedding Lambda → OpenSearch →
Browser / Notebook → API Gateway → Proxy Lambda → Bedrock Agent (Haiku) →
search_reviews action group →
Search Lambda → OpenSearch
```

> **Note:** Cells 2–3 (ETL and Embedding) only need to be run once on initial setup.  
> For the demo, start from **Cell 4** (smoke test).

In [6]:
# =====================================================
# Cell 1 — Configuration
# =====================================================

import os, time, json, boto3, requests

# ---------- CONFIG — update these values ----------
AWS_REGION  = "eu-central-1"
API_URL     = "https://bbtsnacxpf.execute-api.eu-central-1.amazonaws.com/query"
API_KEY     = "tastetrend-demo-2026"

ETL_LAMBDA       = "tastetrend-poc-etl"
EMBEDDING_LAMBDA = "tastetrend-poc-embedding"
ARTIFACTS_BUCKET = f"tastetrend-poc-artifacts-550744777598"

os.environ["AWS_REGION"] = AWS_REGION

session       = boto3.Session(region_name=AWS_REGION)
lambda_client = session.client("lambda")
s3            = session.client("s3")

print("Environment configured")
print(f"   Region : {AWS_REGION}")
print(f"   API URL: {API_URL}")

Environment configured
   Region : eu-central-1
   API URL: https://bbtsnacxpf.execute-api.eu-central-1.amazonaws.com/query


In [ ]:
# =====================================================
# Cell 2 — Trigger ETL Lambda  (run once on setup)
# =====================================================

print("Invoking ETL Lambda...")
resp = lambda_client.invoke(
    FunctionName=ETL_LAMBDA,
    InvocationType="RequestResponse",
    Payload=json.dumps({}).encode()
)
status = resp["StatusCode"]
result = json.loads(resp["Payload"].read())
print(f"ETL status : {status}")
print(f"ETL result : {json.dumps(result, indent=2)}")

# List processed files
objects = s3.list_objects_v2(Bucket=ARTIFACTS_BUCKET, Prefix="processed/")
keys = [obj["Key"] for obj in objects.get("Contents", [])]
print(f"\nProcessed files in S3 ({len(keys)}):")
for k in keys[:5]:
    print(f"  {k}")

In [ ]:
# =====================================================
# Cell 3 — Trigger Embedding Lambda  (run once on setup)
# =====================================================

print("Invoking Embedding Lambda to index documents into OpenSearch...")
resp = lambda_client.invoke(
    FunctionName=EMBEDDING_LAMBDA,
    InvocationType="RequestResponse",
    Payload=json.dumps({}).encode()
)
status = resp["StatusCode"]
result = json.loads(resp["Payload"].read())
print(f"Embedding status : {status}")
print(f"Embedding result : {json.dumps(result, indent=2)}")

In [7]:
# =====================================================
# Cell 4 — Smoke Test via API Gateway
# =====================================================

def ask(query: str, conversation_id: str = None) -> dict:
    """Send a query to the TasteTrend API and return the parsed response."""
    payload = {"query": query}
    if conversation_id:
        payload["conversation_id"] = conversation_id

    start = time.time()
    resp = requests.post(
        API_URL,
        headers={"Content-Type": "application/json", "x-api-key": API_KEY},
        json=payload,
        timeout=60
    )
    elapsed = round((time.time() - start) * 1000, 1)

    if resp.status_code != 200:
        raise RuntimeError(f"API error {resp.status_code}: {resp.text}")

    data = resp.json()
    return {
        "answer": data.get("answer", ""),
        "results": data.get("results", []),
        "conversation_id": data.get("conversation_id"),
        "latency_ms": data.get("latency_ms", elapsed)
    }


print("Running smoke test...")
response = ask("Which location has the most complaints about waiting times?")

print(f"\n Smoke test passed ({response['latency_ms']:.0f}ms)")
print(f"\nAnswer:\n{response['answer']}")
print(f"\nSource reviews returned: {len(response['results'])}")
for r in response["results"][:3]:
    print(f"  [{r.get('restaurant_name')} ★{r.get('rating')}] {r.get('text', '')[:80]}...")

Running smoke test...

 Smoke test passed (4166ms)

Answer:
Based on the review data, the Downtown and Uptown locations appear to have the most complaints about long waiting times and wait times. Customers mention having to wait 1.5 hours or more to be seated, even during off-peak hours, and describe the wait times as "painful", "unnecessary", and "excruciating". There are fewer complaints about waiting times at the Midtown location.

Source reviews returned: 5
  [Downtown ★5.0] Ok yeah, the service can be a little high falutin', and the wait is painful but ...
  [Downtown ★5.0] If you're having a burger craving you need to eat here.  I came here with the hu...
  [Uptown ★0.0] Terrible wait (unnecessary), average service, above average prices, and excrucia...


In [8]:
# =====================================================
# Cell 5 — Evaluation: Example Business Queries
# =====================================================

EVAL_QUERIES = [
    "What is the best restaurant overall?",
    "What is the general consensus of the downtown restaurant?",
    "What do customers like most about the Uptown location?",
    "What do people complain about in the Riverside restaurant?",
    "How does service quality compare between Uptown and Riverside?",
    "Which menu items get the best reviews?",
    "Which location has the most complaints about waiting times?",
    "Do customers think our food is worth the price?",
    "What do customers say about staff friendliness across all locations?",
    "Which location would you recommend to a new customer and why?"
]

results_log = []

print(f"Running evaluation over {len(EVAL_QUERIES)} business queries...\n")
print("-" * 70)

for i, query in enumerate(EVAL_QUERIES, 1):
    try:
        r = ask(query)
        status = "Good"
        answer_preview = r["answer"][:100].replace("\n", " ")
        source_count = len(r["results"])
        latency = r["latency_ms"]
        results_log.append({"query": query, "status": "ok", "latency_ms": latency, "sources": source_count})
    except Exception as e:
        status = "Not passed"
        answer_preview = str(e)
        source_count = 0
        latency = 0
        results_log.append({"query": query, "status": "error", "error": str(e)})

    print(f"{status} [{i:02d}] {query}")
    print(f"       {answer_preview}...")
    print(f"       latency: {latency:.0f}ms | sources: {source_count}")
    print()

# Summary
ok    = sum(1 for r in results_log if r["status"] == "ok")
total = len(results_log)
avg_latency = sum(r.get("latency_ms", 0) for r in results_log if r["status"] == "ok") / max(ok, 1)

print("-" * 70)
print(f"Results  : {ok}/{total} queries succeeded")
print(f"Avg latency: {avg_latency:.0f}ms")

Running evaluation over 10 business queries...

----------------------------------------------------------------------
Good [01] What is the best restaurant overall?
       Based on the reviews, I don't have enough information to definitively identify the single "best" res...
       latency: 7045ms | sources: 5

Good [02] What is the general consensus of the downtown restaurant?
       Based on the reviews, the general consensus on the downtown restaurant seems to be mixed. Some custo...
       latency: 5557ms | sources: 5

Good [03] What do customers like most about the Uptown location?
       Based on the reviews, it seems customers really like the following aspects of the Uptown location:  ...
       latency: 5981ms | sources: 5

Good [04] What do people complain about in the Riverside restaurant?
       I'm sorry, but the search did not return any relevant reviews for the Riverside restaurant. Without ...
       latency: 5645ms | sources: 5

Good [05] How does service quality compa

---
## Cell 6 — Live Prompt Testing

Use this cell during the **demo session** to test arbitrary questions live.  
The agent maintains conversation history within a session — follow-up questions work.

Type `exit` or `quit` to end the session.

In [ ]:
# =====================================================
# Cell 6 — Live Interactive Demo  (CEO mode )
# =====================================================

import uuid
conversation_id = str(uuid.uuid4())
print(f"Session ID: {conversation_id}")
print("Ask anything about TasteTrend restaurant reviews. Type 'exit' to stop.\n")
print("=" * 60)

while True:
    query = input("\nCEO > ").strip()
    if not query or query.lower() in ("exit", "quit"):
        print("Session ended.")
        break
    try:
        r = ask(query, conversation_id=conversation_id)
        print(f"\nTasteTrend AI ({r['latency_ms']:.0f}ms):")
        print(r["answer"])
        if r["results"]:
            print(f"\n  [{len(r['results'])} source reviews retrieved]")
            for rev in r["results"][:2]:
                print(f"  • [{rev.get('restaurant_name')} ★{rev.get('rating')}] {rev.get('text', '')[:100]}...")
        print("-" * 60)
    except Exception as e:
        print(f"  Error: {e}")